# 11장 — AI가 만드는 고리: 모델 붕괴·보상 해킹·플라이휠

웹 챕터(11장)와 **같은 장난감 모형, 같은 수치**를 파이썬으로 계산합니다(난수 없음, 해석적·결정적):
꼬리를 자른 합성 데이터로 거듭 배우는 모델 붕괴, 대리 지표를 쫓는 보상 해킹과 KL 목줄, 데이터 플라이휠의 한계.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch11')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 모델 붕괴(장난감): 세대마다 꼬리를 자르면

이전 모델은 ±2σ 바깥의 드문 사례를 만들지 않는다고 둡니다. 정규분포(3부 6장)를 ±2σ에서 자르면 남는 분산의 비율은

$$c = 1 - \frac{2 \cdot 2\,\varphi(2)}{\Phi(2) - \Phi(-2)} \approx 0.774 \quad(\text{폭} \times 0.880)$$

실제 데이터를 비율 α 만큼 섞으면 $v_{g+1} = \alpha + (1 - \alpha)\,c\,v_g$, 평형은 $v^* = \alpha / (1 - (1-\alpha)c)$.

In [ ]:
import math

phi = lambda x: math.exp(-x * x / 2) / math.sqrt(2 * math.pi)
Phi = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))
c = 1 - 2 * 2 * phi(2) / (Phi(2) - Phi(-2))
print(f'±2σ 절단의 분산 비율 c = {c:.4f},  폭 비율 √c = {math.sqrt(c):.4f}')

def sigmas(alpha, gens=20):
    v, out = 1.0, [1.0]
    for _ in range(gens):
        v = alpha + (1 - alpha) * c * v
        out.append(math.sqrt(v))
    return out

def tail_share(sigma):
    # 폭 σ 인 분포가 원래(폭 1)의 꼬리 |x| > 2 를 만들어 내는 비율
    return (1 - Phi(2 / sigma)) / (1 - Phi(2))

for a in [0, 0.1, 0.3, 0.5]:
    s = sigmas(a)
    lim = math.sqrt(a / (1 - (1 - a) * c)) if a > 0 else 0.0
    print(f'실제 데이터 {a * 100:3.0f}%:  1세대 {s[1]:.3f}  5세대 {s[5]:.3f}  10세대 {s[10]:.3f}  20세대 {s[20]:.3f}   평형 폭 {lim:.3f}')

s0 = sigmas(0)
assert round(math.sqrt(c), 3) == 0.880 and round(c, 3) == 0.774
assert round(s0[2], 3) == 0.774 and round(s0[5], 3) == 0.527 and round(s0[10], 3) == 0.277 and round(s0[20], 3) == 0.077
assert round(sigmas(0.1)[20], 3) == 0.574 and round(sigmas(0.3)[20], 3) == 0.809 and round(sigmas(0.5)[20], 3) == 0.903
print(f'10세대 뒤 원래 꼬리가 남은 정도: {tail_share(s0[10]) * 100:.4f}%')
assert tail_share(s0[10]) < 1e-6
print('✅ 합성만: 10세대 폭 0.277 · 실제 10% → 0.574, 30% → 0.809 에서 멈춤')

## 2. 보상 해킹(장난감): 대리 지표를 쫓는 고리와 KL 목줄

최적화 강도 e 가 '보상 − β·e²/2'(KL 목줄)를 키우는 쪽으로 걷습니다: $e \leftarrow e + 0.1(1 - \beta e)$, 처음 e = 0.
보상 모델 점수(대리 지표) = e, 진짜 품질 = $e - 0.1e^2$ (e = 5 에서 최고 2.5). 결국 $e = 1/\beta$ 에 멈추는 균형 루프예요.

In [ ]:
def reward_run(beta, steps=300):
    e, out = 0.0, []
    for k in range(steps + 1):
        out.append((k, e, e - 0.1 * e * e))
        e = e + 0.1 * (1 - beta * e)
    return out

for beta in [0, 0.05, 0.1, 0.2, 0.5, 1.0]:
    r = reward_run(beta)
    pk = max(r, key=lambda x: (x[2], -x[0]))
    print(f'β = {beta:4.2f}:  300걸음 점수 {r[-1][1]:5.2f}, 진짜 품질 {r[-1][2]:6.2f}   (품질 정점 {pk[2]:.2f} @ {pk[0]}걸음)')

r05 = reward_run(0.05)
pk = max(r05, key=lambda x: (x[2], -x[0]))
assert pk[0] == 57 and round(pk[2], 2) == 2.5 and round(r05[-1][2], 2) == -8.64
assert round(reward_run(0.2)[-1][2], 2) == 2.5
assert round(reward_run(1.0)[-1][2], 2) == 0.9
assert round(reward_run(0)[-1][2]) == -60
print('✅ β = 0.05: 57걸음에 품질 2.5 정점 → 300걸음 −8.64 · β = 0.2: 2.5 · β = 1: 0.9')

## 3. 데이터 플라이휠: 강화 루프만 그린 계획 vs 한계를 넣은 실제

처음 100명, 성장률 월 0.5, 시장 1만 명. 계획은 $100 e^{0.5t}$, 실제는 로지스틱(7장).

In [ ]:
U0, M, r = 100, 10000, 0.5
plan = lambda t: U0 * math.exp(r * t)
real = lambda t: M / (1 + (M - U0) / U0 * math.exp(-r * t))
for t in [3, 6, 9, 12, 18, 24]:
    print(f'{t:2d}개월  계획 {plan(t):12,.0f}명   실제 {real(t):7,.0f}명')
assert round(plan(12)) == 40343 and round(real(12)) == 8030
print('✅ 12개월: 계획 40,343명(시장 1만 명을 넘음) vs 실제 8,030명')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for a, col in [(0, '#7c3aed'), (0.1, '#a78bfa'), (0.3, '#059669')]:
    axes[0].plot(range(21), sigmas(a), marker='o', ms=3, color=col, label=f'실제 데이터 {a * 100:.0f}%')
axes[0].set_xlabel('세대'); axes[0].set_ylabel('분포의 폭'); axes[0].set_title('모델 붕괴: 실제 데이터가 브레이크'); axes[0].legend(); axes[0].grid(alpha=0.3)
for beta, col in [(0.05, '#e11d48'), (0.2, '#059669'), (1.0, '#64748b')]:
    rr = reward_run(beta)
    axes[1].plot([x[0] for x in rr], [x[2] for x in rr], color=col, label=f'진짜 품질 β = {beta}')
axes[1].plot([x[0] for x in r05], [x[1] for x in r05], '--', color='#d97706', label='점수(β = 0.05)')
axes[1].set_ylim(-10, 20); axes[1].set_xlabel('학습 걸음'); axes[1].set_title('보상 해킹: 목줄의 세기'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
ts = [i * 0.25 for i in range(97)]
axes[2].plot(ts, [plan(t) for t in ts], '--', color='#f43f5e', label='강화 루프만 그린 계획')
axes[2].plot(ts, [real(t) for t in ts], color='#059669', label='한계를 넣은 실제')
axes[2].axhline(M, ls=':', color='#2563eb')
axes[2].set_ylim(0, 14000); axes[2].set_xlabel('개월'); axes[2].set_title('데이터 플라이휠'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 오늘의 발견

- **모델 붕괴**: 꼬리를 자른 합성 데이터만으로 배우면 폭이 세대마다 × 0.880(정확히는 0.8796, 10세대 0.277). 같은 비율로 줄어드니 목표가 0인 균형 루프 꼴(4장).
- **실제 데이터**는 그 목표를 0에서 끌어올리는 브레이크: 10% → 0.574, 30% → 0.809 에서 평형.
- **보상 해킹**: 점수(대리 지표)를 쫓는 고리는 진짜 품질과 갈라진 뒤에도 계속 민다(β = 0.05: 57걸음 정점 → 300걸음 −8.64).
- **KL 목줄**은 두 번째 균형 루프, 멈추는 자리 e = 1/β 를 정한다. 짧아도(β = 1 → 0.9) 길어도 문제, 알맞으면 2.5.
- **데이터 플라이휠**은 강화 루프지만 수확 체감·포화라는 균형 루프와 함께 그려야 한다(12개월 계획 40,343 vs 실제 8,030).

이제 캡스톤(12장)에서 사내 AI 도입을 처음부터 모형으로 돌려 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**